# Dual-Band TX + Acquisition - ADRV9026

Transmit **two** waveforms on two TX channels at once and capture both ORx
channels (each ORx captured separately).

**How to use:** edit the *Parameters* cell, then *Run All*. The session stays open
between cells; the last cell forces TX safe and disconnects.

> Note: the two ORx ADCs are ORx1/ORx2 -> ADC0 and ORx3/ORx4 -> ADC1. Capturing
> two bands at once therefore uses one front-end from each ADC (e.g. ORx2 + ORx3).

## 1. Parameters (edit me)

**Level ranges**
- `TX_ATTEN_DB` - TX attenuation, **0.0 to 41.95 dB** in 0.05 dB steps. `0` = no attenuation (max power / strongest), `41.95` = max attenuation (weakest, the safe-state value). Higher number = quieter TX.
- `ORX_GAIN_INDEX` - ORx gain-table index, **0 to 255**, higher = more gain. Set manually below (the programmed default is 195; ADI examples use ~240). Raise it toward 255 for a hotter capture, lower it if the capture rails.


In [ ]:
from adrvtrx import TxChannel, RxChannel

# --- Profile -----------------------------------------------------------------
# Pick a friendly key below, OR paste any ".profile" filename / absolute path
# into PROFILE. The filename is resolved under the TES profiles folder.
PROFILES = {
    "98_linksharing":  "ADRV9025Init_StdUseCase98_LinkSharing.profile",   # Tx 491.52 MSPS, Np=12
    "102_linksharing": "ADRV9025Init_StdUseCase102_LinkSharing.profile",  # Np=16
    "14_linksharing":  "ADRV9025Init_StdUseCase14_LinkSharing.profile",
    "50_linksharing":  "ADRV9025Init_StdUseCase50_LinkSharing.profile",
}
PROFILE = "98_linksharing"          # <-- choose here

# --- Bands: one (TX, ORx, signal) per band. Bench wiring: TX2->ORx2, TX3->ORx3.
INPUT_DIR = "C:/Users/ohammi/OneDrive - aus.edu/DualBand/input_100"
BANDS = [
    {"name": "band1", "tx": TxChannel.TX2, "orx": RxChannel.ORX2, "signal": f"{INPUT_DIR}/Signal1.txt"},
    {"name": "band2", "tx": TxChannel.TX3, "orx": RxChannel.ORX3, "signal": f"{INPUT_DIR}/Signal2.txt"},
]

# --- Levels / capture --------------------------------------------------------
TX_ATTEN_DB = 10.0    # 0.0-41.95 dB; 0 = full power, higher = quieter
CAPTURE_MS  = 0.1     # snapshot length in ms
NORMALIZE   = True    # scale each waveform to full scale for the profile's bit depth

# --- ORx gain: software AGC (auto-level, per band) or manual -----------------
USE_AGC         = True    # True = auto-level each ORx; False = manual ORX_GAIN_INDEX
ORX_TARGET_DBFS = -1.0    # AGC target for the captured peak (dBFS)
ORX_TOL_UP_DB   = 0.3     # accept up to target + this (toward the rail)
ORX_TOL_DOWN_DB = 0.6     # accept down to target - this (toward the floor)
ORX_GAIN_INDEX  = 195     # manual ORx gain index, 0-255 -- used ONLY when USE_AGC=False
SAVE_DIR    = None    # e.g. "captures/dualband"; None = do not save
# --- LOs (center frequencies) -----------------------------------------------
# Per-pair mapping (config): band A = Rx1/2+Tx1/2+ORx1/2 -> LO1 ; band B =
# Rx3/4+Tx3/4+ORx3/4 -> LO2 (ORx follows its TX). None = keep the config default.
LO1_HZ = None   # band A center (Hz)
LO2_HZ = None   # band B center (Hz)


## 2. Imports, load config, choose profile

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from adrvtrx.config import load_config
from adrvtrx.profile import read_profile
from adrvtrx.radio import Radio
from adrvtrx.experiment import verify_status
from adrvtrx.waveform import load_tab_iq, full_scale
from adrvtrx.transmit import transmit_bands
from adrvtrx.capture import capture, measure_delay, autolevel_capture
from adrvtrx.gain import clip_report
from adrvtrx._enums import is_orx

cfg = load_config()
cfg.profile_name = PROFILES.get(PROFILE, PROFILE)   # friendly key or raw filename/path
if LO1_HZ is not None:
    cfg.lo.lo1_hz = int(LO1_HZ)
if LO2_HZ is not None:
    cfg.lo.lo2_hz = int(LO2_HZ)
print(f"LO1 (band A, Tx1/2) = {cfg.lo.lo1_hz/1e6:.1f} MHz | LO2 (band B, Tx3/4) = {cfg.lo.lo2_hz/1e6:.1f} MHz")
info = read_profile(cfg.profile_path)
print("Profile:", cfg.profile_path.name)

## 3. Signal-path summary - sample rates & bit depth

In [ ]:
# Sample rate + bit depth per signal path. Prepare your TX waveform at the TX
# rate/bit-depth; a capture comes back at its path's rate (ORx runs faster than
# the main Rx in link-sharing profiles). full scale = 2**(Np-1) - 1.
def _fs(bits):
    return (1 << (bits - 1)) - 1

print(f"{'path':<6}{'rate (MSPS)':>14}{'bits (Np)':>12}{'full scale':>12}")
print(f"{'TX':<6}{info.tx_rate_khz/1000:>14.3f}{info.tx_bits:>12}{_fs(info.tx_bits):>12}")
print(f"{'Rx':<6}{info.rx_rate_khz/1000:>14.3f}{info.rx_bits:>12}{_fs(info.rx_bits):>12}")
print(f"{'ORx':<6}{info.orx_rate_khz/1000:>14.3f}{info.rx_bits:>12}{_fs(info.rx_bits):>12}")

## 4. Load the input waveforms

In [ ]:
for b in BANDS:
    b["wave"] = load_tab_iq(b["signal"])
    print(f"{b['name']}: {b['tx'].name} -> {b['orx'].name} | "
          f"{len(b['wave'])} samples from {b['signal']}")

# All bands must be the same length (they transmit together).
lengths = {len(b["wave"]) for b in BANDS}
assert len(lengths) == 1, f"waveforms differ in length: {lengths}"

## 5. Connect, program, verify

In [ ]:
# Opens the session and LEAVES IT OPEN for the cells below. TX is forced safe on
# connect; programming applies the profile, LOs, masks and tx->orx mapping.
# Run the final "Safe-state & disconnect" cell when you are done.
radio = Radio(cfg)
radio.connect()
radio.force_safe()
radio.program()
for k, v in verify_status(radio).items():
    print(f"{k}: {v}")

## 6. Transmit both bands + capture one aligned snapshot

In [ ]:
# Set TX levels, then transmit both bands (continuous) so each ORx sees signal.
for b in BANDS:
    radio.set_tx_atten(b["tx"], TX_ATTEN_DB)

transmit_bands(radio, {b["tx"]: b["wave"] for b in BANDS}, info.tx_bits,
               continuous=True, do_normalize=NORMALIZE)

# Per-band ORx gain: auto-level (AGC) each band's ORx, or set the manual index.
for b in BANDS:
    if USE_AGC:
        agc = autolevel_capture(radio, b["orx"], bits=info.rx_bits, orx_rate_hz=info.orx_rate_hz,
                                target_dbfs=ORX_TARGET_DBFS, tol_up_db=ORX_TOL_UP_DB,
                                tol_down_db=ORX_TOL_DOWN_DB)
        print(f"{b['name']} {b['orx'].name}: AGC gain {agc.final_gain_index} | "
              f"peak {agc.final_peak_dbfs:+.2f} dBFS | railed {agc.railed} | {agc.reason}")
    else:
        radio.set_rx_gain(b["orx"], ORX_GAIN_INDEX)   # manual ORx gain (per band)

# Per-band capture: enable ONLY that band's ORx for its own snapshot (enabling two
# ORx at once mis-routes on this hardware). Measure each TX->ORx delay while TX live.
for b in BANDS:
    b["cap"] = capture(radio, int(b["orx"]), CAPTURE_MS, bits=info.rx_bits).channels[b["orx"]]
    d_s, d_ns, corr = measure_delay(radio, b["orx"], b["wave"], bits=info.rx_bits, fs=info.orx_rate_hz)
    b["delay_samples"], b["delay_ns"], b["corr"] = d_s, d_ns, corr
    print(f"{b['name']} {b['orx'].name}: {len(b['cap'].i)} samples | "
          f"delay {d_s:.3f} samp ({d_ns:.1f} ns), corr {corr:.3f}")

radio.disable_tx()


## 7. Inspect each band (time + spectrum)

In [ ]:
fig, axes = plt.subplots(len(BANDS), 2, figsize=(12, 3.4 * len(BANDS)))
if len(BANDS) == 1:
    axes = np.array([axes])

for row, b in enumerate(BANDS):
    cap, ch = b["cap"], b["orx"]
    rate = info.orx_rate_hz if is_orx(ch) else info.rx_rate_hz
    iq = cap.i.astype(float) + 1j * cap.q.astype(float)
    rep = clip_report(cap.i, cap.q, info.rx_bits)
    print(f"{b['name']} {ch.name}: peak {rep.peak_dbfs:+.1f} dBFS | railed {rep.railed_samples}")

    fsfull = full_scale(info.rx_bits)
    n = len(iq)
    t_us = np.arange(n) / rate * 1e6
    X = np.fft.fftshift(np.fft.fft(iq * np.hanning(n)))
    fmhz = np.fft.fftshift(np.fft.fftfreq(n, 1.0 / rate)) / 1e6

    axes[row, 0].plot(t_us, iq.real / fsfull, label="I")
    axes[row, 0].plot(t_us, iq.imag / fsfull, label="Q")
    axes[row, 0].set_title(f"{b['name']} {ch.name} - time")
    axes[row, 0].set_xlabel("us"); axes[row, 0].grid(True); axes[row, 0].legend(loc="upper right")
    axes[row, 1].plot(fmhz, 20 * np.log10(np.abs(X) / np.max(np.abs(X)) + 1e-12))
    axes[row, 1].set_title(f"{b['name']} {ch.name} - spectrum")
    axes[row, 1].set_xlabel("MHz"); axes[row, 1].grid(True)

plt.tight_layout(); plt.show()

## 8. (optional) Save the captures

In [ ]:
if SAVE_DIR:
    from pathlib import Path
    Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
    for b in BANDS:
        p = f"{SAVE_DIR}/{b['name']}_{b['orx'].name}.txt"
        b["cap"].save(p)
        print("saved", p)
else:
    print("SAVE_DIR is None - not saving")

## 9. Safe-state & disconnect (run when finished)

In [ ]:
radio.safe_state()     # max attenuation + clear TX enable
radio.disconnect()
print("TX safe, board disconnected")